In [6]:
import requests
from bs4 import BeautifulSoup
import re
import time
import pandas as pd
from IPython.display import display

In [15]:


def get_apartment_links(page_url):
    """Збирає посилання на всі квартири з однієї сторінки пошуку."""
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    response = requests.get(page_url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    links = set()
    elements = soup.find_all(attrs={"data-event-options": re.compile(r"page_id:\d+")})
    for el in elements:
        match = re.search(r'page_id:(\d+)', el.get('data-event-options', ''))
        if match:
            links.add(f"https://lun.ua/realty/{match.group(1)}")
            
    return list(links)

def parse_apartment_data(url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    data = {
        'url': url, 'price': None, 'address': None, 'rooms': None, 
        'area_total': None, 'area_living': None, 'area_kitchen': None,
        'floor': None, 'total_floors': None, 
        'build_year': None, 'construction_tech': None, 'heating_type': None,
        'building_details': [], 'commission': None, 'description': None
    }
    
    # Словник для точного зведення технології будівництва до твого списку
    tech_mapping = {
        'моноліт': 'монолітно-каркасна',
        'утеплена панель': 'утеплена панель',
        'панель': 'панельна технологія', # спрацює, якщо не "утеплена"
        'цегл': 'цегляна технологія',
        'блок': 'блочна технологія',
        'блочн': 'блочна технологія'
    }
    
    # 1. ЦІНА
    price_tag = soup.find(class_=re.compile("RealtyDetails_priceMain|RealtyCard_price"))
    if price_tag:
        data['price'] = price_tag.text.strip()
    else:
        og_title = soup.find("meta", property="og:title")
        title_text = og_title["content"] if og_title else (soup.title.text if soup.title else "")
        if title_text:
            price_match = re.search(r'((?:[\$€]\s*)?[\d\s\u202f\u200a]+(?:грн|\$|€)?)', title_text)
            if price_match: data['price'] = price_match.group(1).strip()

    # 2. АДРЕСА
    address_tag = soup.find(class_=re.compile("RealtyDetails_address|Realty_address"))
    if address_tag: data['address'] = address_tag.text.strip()

    # 3. ОПИС
    desc_tag = soup.find("article") or soup.find(attrs={"itemprop": "description"}) or soup.find(class_=re.compile("description|Description|ExpandableText_text"))
    if desc_tag: data['description'] = desc_tag.text.strip()
    
    # 4. КОМІСІЯ (Лейбли)
    labels = soup.find_all(class_=re.compile("LabelRefresh-module_content|LabelList_label"))
    for label in labels:
        if 'без комісії' in label.text.lower():
            data['commission'] = 'без комісії'
            break

    # 5. ХАРАКТЕРИСТИКИ
    properties = soup.find_all(class_=re.compile("PropertyItem|Characteristic|param"))
    for prop in properties:
        text = prop.text.strip()
        if not text or text in data['building_details']: continue
        data['building_details'].append(text)
        
        text_lower = text.lower()
        
        # Комісія
        if 'комісі' in text_lower and not data['commission']:
            if 'без' in text_lower:
                data['commission'] = 'без комісії'
            elif '%' in text_lower or re.search(r'\d+', text_lower):
                data['commission'] = text
        
        # Кімнати
        if 'кімнат' in text_lower and not data['rooms']: 
            rooms_match = re.search(r'\d+', text)
            if rooms_match: data['rooms'] = rooms_match.group(0)
                
        # Площа
        elif 'м²' in text and not data['area_total']: 
            area_clean = text.replace('м²', '').strip()
            area_parts = [p.strip() for p in area_clean.split('/')]
            if len(area_parts) >= 1: data['area_total'] = area_parts[0]
            if len(area_parts) >= 2: data['area_living'] = area_parts[1]
            if len(area_parts) >= 3: data['area_kitchen'] = area_parts[2]
                
        # Поверх
        elif 'поверх' in text_lower and not data['floor']:
            floor_match = re.search(r'поверх (\d+) з (\d+)', text)
            if floor_match:
                data['floor'] = floor_match.group(1)
                data['total_floors'] = floor_match.group(2)
            else:
                data['floor'] = text
                
        # Рік будівництва
        elif 'рік будівництва' in text_lower or 'рік побудови' in text_lower:
            year_match = re.search(r'\d{4}', text)
            if year_match: data['build_year'] = year_match.group(0)
                
        # Тип опалення 
        elif 'опалення' in text_lower and 'без світла' not in text_lower:
            data['heating_type'] = text
            
        # Технологія будівництва (шукаємо за словником)
        if not data['construction_tech']:
            for key, exact_value in tech_mapping.items():
                if key in text_lower:
                    # Якщо знайшли "панель", перевіримо чи це не "утеплена панель"
                    if key == 'панель' and 'утеплена панель' in text_lower:
                        continue 
                    data['construction_tech'] = exact_value
                    break

    # 6. КОМІСІЯ ТА ТЕХНОЛОГІЯ В ОПИСІ (Резервний пошук, якщо не знайшли в характеристиках)
    if data['description']:
        desc_lower = data['description'].lower()
        
        if not data['commission']:
            comm_match = re.search(r'(?i)(?:комісі|комісійні).*?(\d+\s*%|без комісії)', data['description'])
            if comm_match: data['commission'] = comm_match.group(0)
                
        if not data['construction_tech']:
            for key, exact_value in tech_mapping.items():
                if key in desc_lower:
                    if key == 'панель' and 'утеплена панель' in desc_lower:
                        continue 
                    data['construction_tech'] = exact_value
                    break
            
    data['building_details'] = " | ".join(data['building_details'])
    return data

def run_scraper(pages_list, limit_flats=None):
    """
    Головна функція. Приймає список сторінок і повертає готовий DataFrame.
    limit_flats - обмежує кількість зібраних квартир для тестів (напр. 5).
    """
    all_links = []
    print("Шукаємо посилання на квартири...")
    for page in pages_list:
        links = get_apartment_links(page)
        all_links.extend(links)
        print(f"Знайдено {len(links)} на {page}")
        time.sleep(1) 
        
    unique_links = list(set(all_links))
    if limit_flats:
        unique_links = unique_links[:limit_flats]
        
    print(f"\nПочинаємо збір даних по {len(unique_links)} квартирах...")
    scraped_data = []
    
    for i, link in enumerate(unique_links):
        print(f"[{i+1}/{len(unique_links)}] {link}")
        scraped_data.append(parse_apartment_data(link))
        time.sleep(0.5)
        
    print("\nГотово!")
    return pd.DataFrame(scraped_data)

# ==========================================
# ЗАПУСК КОДУ:
# ==========================================
pages_to_scrape = [
    "https://lun.ua/rent/kyiv/flats?page=11",
    "https://lun.ua/rent/kyiv/flats?page=13"
]

# limit_flats=5 - це для швидкого тесту. Щоб зібрати всі, просто прибери цей параметр.
df_result = run_scraper(pages_to_scrape, limit_flats=5)

# Виводимо таблицю
display(df_result)

Шукаємо посилання на квартири...
Знайдено 24 на https://lun.ua/rent/kyiv/flats?page=11
Знайдено 24 на https://lun.ua/rent/kyiv/flats?page=13

Починаємо збір даних по 5 квартирах...
[1/5] https://lun.ua/realty/4655259712
[2/5] https://lun.ua/realty/4692442209
[3/5] https://lun.ua/realty/4692442180
[4/5] https://lun.ua/realty/4691874103
[5/5] https://lun.ua/realty/4692442197

Готово!


,url,price,address,rooms,area_total,area_living,area_kitchen,floor,total_floors,build_year,construction_tech,heating_type,building_details,commission,description
0,https://lun.ua/realty/4655259712,30 000 грн,"Салютна вулиця, 2к19",1,34.2,10,14,6,9,2022,блочна технологія,централізоване опалення,1 кімната | 34.2 / 10 / 14 м² | поверх 6 з 9 |...,без комісії,Пропонується стильна та затишна квартира у пре...
1,https://lun.ua/realty/4692442209,23 957 грн,"бульвар Миколи Руденка, 14-Е",2,80,-,13,7,15,2014,цегляна технологія,централізоване опалення,2 кімнати | 80 / - / 13 м² | поверх 7 з 15 | з...,NaN,Квартира має роздільне планування. Стан ремонт...
2,https://lun.ua/realty/4692442180,18 000 грн,"Березняківська вулиця, 14-А",2,60,-,9,9,9,1970,панельна технологія,централізоване опалення,2 кімнати | 60 / - / 9 м² | поверх 9 з 9 | з р...,NaN,Пропонується до довгострокової оренди чудова т...
3,https://lun.ua/realty/4691874103,11 000 грн,"вулиця Валерія Лобановського, 9",1,44,-,11,1,15,2016,цегляна технологія,централізоване опалення,1 кімната | 44 / - / 11 м² | поверх 1 з 15 | з...,NaN,Квартира має роздільне планування та косметичн...
4,https://lun.ua/realty/4692442197,$ 1 000,"Голосіївський проспект, 60",2,65,-,15,19,25,2015,монолітно-каркасна,централізоване опалення,2 кімнати | 65 / - / 15 м² | поверх 19 з 25 | ...,NaN,Пропонується простора квартира зі зручним план...
